# Proyek Pengembangan dan Pengoperasian Sistem Machine Learning
**Nama:** Muhammad Reza Pahlevi Harahap  
**Username:** `reza_harahap`

Notebook ini **benar-benar menjalankan seluruh pipeline TFX melalui Apache Beam**. Output eksekusi disimpan di notebook sebagai bukti reviewer.


## 1. Environment dan Dataset
Pipeline dijalankan dengan TFX 1.15.1 / TensorFlow 2.15.x di Python 3.10.


In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import tensorflow as tf
import tfx

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "breast_cancer.csv"
df = pd.read_csv(DATA_PATH)
print("Project root       :", PROJECT_ROOT)
print("Python             :", sys.version.split()[0])
print("TFX version        :", tfx.__version__)
print("TensorFlow version :", tf.__version__)
print("Dataset exists     :", DATA_PATH.exists())
print("Dataset shape      :", df.shape)
print("Jumlah fitur       :", len(df.columns) - 1)
print("Distribusi label   :", df["label"].value_counts().sort_index().to_dict())


## 2. Inisialisasi Seluruh Komponen TFX
`create_pipeline()` menyusun `CsvExampleGen`, `StatisticsGen`, `SchemaGen`, `ExampleValidator`, `Transform`, `Tuner`, `Trainer`, `Resolver`, `Evaluator`, dan `Pusher` menjadi satu objek `pipeline.Pipeline`.


In [ ]:
from pipeline import create_pipeline, PIPELINE_ROOT, SERVING_MODEL_DIR, METADATA_PATH

beam_pipeline = create_pipeline()
print("Pipeline name :", beam_pipeline.pipeline_info.pipeline_name)
print("Pipeline root :", PIPELINE_ROOT)
print("Metadata path :", METADATA_PATH)
print("Serving dir   :", SERVING_MODEL_DIR)
print("Komponen:")
for i, component in enumerate(beam_pipeline.components, 1):
    print(f"{i:02d}. {component.id}")


## 3. Eksekusi Pipeline dengan `BeamDagRunner`
Cell berikut adalah eksekusi pipeline TFX yang sebenarnya, **bukan pemeriksaan teks atau placeholder artifact**.


In [ ]:
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner

print("Starting BeamDagRunner...")
BeamDagRunner().run(pipeline=beam_pipeline)
print("TFX PIPELINE EXECUTION COMPLETED SUCCESSFULLY")


## 4. Verifikasi Artifact TFX dan Hasil Tuner


In [ ]:
from pathlib import Path
import json

root = Path(PIPELINE_ROOT)
required = ["CsvExampleGen", "StatisticsGen", "SchemaGen", "ExampleValidator", "Transform", "Tuner", "Trainer", "Evaluator", "Pusher"]
print("Pipeline root exists:", root.exists())
print("Total artifact files:", sum(p.is_file() for p in root.rglob("*")))
for name in required:
    component_dir = root / name
    files = [p for p in component_dir.rglob("*") if p.is_file()] if component_dir.exists() else []
    print(f"{name:18s}: {len(files)} artifact file(s)")

hp_files = list((root / "Tuner").rglob("best_hyperparameters.txt"))
if not hp_files:
    raise FileNotFoundError("best_hyperparameters.txt tidak ditemukan")
hp_path = max(hp_files, key=lambda p: p.stat().st_mtime)
hp = json.loads(hp_path.read_text())
print("Best hyperparameters:", hp["values"])

blessed = list((root / "Evaluator").rglob("BLESSED"))
print("Evaluator BLESSED:", bool(blessed))
if not blessed:
    raise RuntimeError("Model tidak BLESSED")

saved = list(Path(SERVING_MODEL_DIR).rglob("saved_model.pb"))
print("Pusher SavedModel:", saved)
if not saved:
    raise RuntimeError("Pusher SavedModel tidak ditemukan")


## 5. Nilai BinaryAccuracy dan AUC dari TFX Evaluator


In [ ]:
import tensorflow_model_analysis as tfma

eval_root = Path(PIPELINE_ROOT) / "Evaluator" / "evaluation"
metric_files = list(eval_root.rglob("metrics-*.tfrecord"))
if not metric_files:
    raise FileNotFoundError("TFX Evaluator metrics tidak ditemukan")
eval_dir = max((p.parent for p in metric_files), key=lambda p: p.stat().st_mtime)
eval_result = tfma.load_eval_result(str(eval_dir))
metrics = eval_result.get_metrics_for_slice(())
validation = tfma.load_validation_result(str(eval_dir))
print("Evaluator directory:", eval_dir)
print("Evaluator metrics (overall slice):")
print(metrics)
print("Evaluator validation_ok:", validation.validation_ok)


## 6. Serving Signature Model Hasil Pusher
Model yang dideploy harus menerima serialized `tf.Example` melalui input `examples` bertipe `DT_STRING`.


In [ ]:
saved_model_pb = max(saved, key=lambda p: p.stat().st_mtime)
model_dir = saved_model_pb.parent
loaded = tf.saved_model.load(str(model_dir))
sig = loaded.signatures["serving_default"]
print("SavedModel:", model_dir)
print("serving_default input:", sig.structured_input_signature)
print("serving_default outputs:", sig.structured_outputs)


## 7. Status Akhir


In [ ]:
print("ALL REVIEWER-CRITICAL TFX CHECKS PASSED")
print("Notebook ini telah menjalankan BeamDagRunner, Tuner, Trainer, Evaluator, dan Pusher secara nyata.")
